# Discount Effectiveness Analysis

> **Goal:** Understand how discounts affect sales volume, revenue, and profit margin.
> Compare discounted vs non-discounted orders, identify discount bands where margin collapses,
> and surface category-level patterns that reveal whether discounting is helping or hurting
> overall business performance.

| Dimension | Key Question |
|---|---|
| Volume | Do discounts drive more units / orders? |
| Revenue | Does higher volume offset the price cut? |
| Margin | At what depth does profit turn negative? |
| Category | Which categories benefit vs bleed under discounts? |
| Optimality | Is there a sweet spot discount range? |


## Project OverviewThis notebook evaluates whether discounting improves retail performance or mainly gives away margin. It uses the Sample Superstore workbook already present in the repository and studies discount depth, revenue, profit, customer segment behaviour, and product-level variation.## Learning Objectives- Measure how discounting changes revenue, volume, and profit together.- Distinguish between discount-driven growth and margin destruction.- Compare discount behaviour across categories and customer segments.- Translate descriptive discount analysis into pricing and promotion decisions.## Business / Research ProblemRetail teams need to know when discounts stimulate useful demand and when they simply erode margin. The core question is whether the observed sales lift is large enough to justify the profitability cost of discounting.## Why This Analysis MattersDiscounts are easy to launch and hard to evaluate well. A structured analysis helps teams separate healthy promotional leverage from recurring margin leakage.

## Dataset OverviewThis notebook uses the `Orders` sheet from the repo-local `Sample - Superstore.xls` workbook. Key fields used here include `Sales`, `Profit`, `Quantity`, `Discount`, `Segment`, `Category`, `Sub-Category`, `Order Date`, and geography fields that support retail performance comparisons.## Dataset Source and License NotesThe workbook is a repo-local copy of the widely circulated Sample Superstore training dataset. Confirm redistribution terms for the exact external source copy you use outside this repository. This notebook only analyzes the local workbook already available in the workspace.

## 1. Environment Setup


## Imports

In [ ]:
import warnings, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# aesthetics
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.titleweight": "bold"})

SEED = 42
np.random.seed(SEED)
print("Libraries loaded OK")


## Configuration / Constants

In [ ]:
ROOT      = pathlib.Path(r"Time Series Analysis/Time Series Forecasting/Sample - Superstore.xls").parent.parent  # workspace root
XLS_PATH  = ROOT / r"Time Series Analysis/Time Series Forecasting/Sample - Superstore.xls"
OUT_DIR   = ROOT / "Data Analysis" / "Discount Effectiveness Analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DISCOUNT_BINS   = [-0.001, 0.0, 0.10, 0.20, 0.30, 0.50, 1.01]
DISCOUNT_LABELS = ["No Discount", "1-10%", "11-20%", "21-30%", "31-50%", ">50%"]

print(f"Dataset : {XLS_PATH}")
print(f"Exists  : {XLS_PATH.exists()}")


## Data Validation Checks

In [ ]:
df = pd.read_excel(XLS_PATH, sheet_name="Orders", engine="xlrd")

# Normalise column names
df.columns = df.columns.str.strip()

# Derived columns
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Year"]       = df["Order Date"].dt.year
df["Month"]      = df["Order Date"].dt.month
df["YearMonth"]  = df["Order Date"].dt.to_period("M")
df["Margin Pct"] = df["Profit"] / df["Sales"].replace(0, np.nan) * 100

print(f"Shape : {df.shape}")
print(f"Date range : {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"Discount range : {df['Discount'].min():.0%} – {df['Discount'].max():.0%}")
df[["Sales","Quantity","Discount","Profit","Margin Pct"]].describe().round(2)


## Data Cleaning

In [ ]:
df["Discount Band"] = pd.cut(
    df["Discount"],
    bins=DISCOUNT_BINS,
    labels=DISCOUNT_LABELS,
    right=True
)

band_counts = df["Discount Band"].value_counts().reindex(DISCOUNT_LABELS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# bar – order count per band
axes[0].bar(band_counts.index, band_counts.values,
            color=sns.color_palette("muted", len(DISCOUNT_LABELS)))
axes[0].set_title("Order Count by Discount Band")
axes[0].set_xlabel("Discount Band")
axes[0].set_ylabel("Number of Orders")
for bar, v in zip(axes[0].patches, band_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f"{v:,}", ha="center", va="bottom", fontsize=9)

# histogram of the raw Discount column
axes[1].hist(df["Discount"], bins=30, color="#4878d0", edgecolor="white")
axes[1].set_title("Raw Discount Value Distribution")
axes[1].set_xlabel("Discount (0 = no discount, 1 = 100%)")
axes[1].set_ylabel("Order Count")
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig(OUT_DIR / "discount_distribution.png", bbox_inches="tight")
plt.show()
print(band_counts.to_string())


## Exploratory Data Analysis
Split the dataset into two groups and compare on key business metrics.


In [ ]:
df["Is Discounted"] = df["Discount"] > 0

summary = df.groupby("Is Discounted").agg(
    Order_Count   = ("Order ID", "count"),
    Total_Sales   = ("Sales", "sum"),
    Total_Profit  = ("Profit", "sum"),
    Avg_Sales     = ("Sales", "mean"),
    Avg_Discount  = ("Discount", "mean"),
    Avg_Margin    = ("Margin Pct", "mean"),
    Median_Margin = ("Margin Pct", "median"),
).rename(index={False: "Non-Discounted", True: "Discounted"})

summary["Revenue Share %"] = (summary["Total_Sales"] / summary["Total_Sales"].sum() * 100).round(1)
summary["Profit Share %"]  = (summary["Total_Profit"] / summary["Total_Profit"].sum() * 100).round(1)
summary["Avg_Discount"]    = summary["Avg_Discount"].map("{:.1%}".format)
summary["Total_Sales"]     = summary["Total_Sales"].map("${:,.0f}".format)
summary["Total_Profit"]    = summary["Total_Profit"].map("${:,.0f}".format)
summary["Avg_Sales"]       = summary["Avg_Sales"].map("${:,.2f}".format)
summary["Avg_Margin"]      = summary["Avg_Margin"].map("{:.1f}%".format)
summary["Median_Margin"]   = summary["Median_Margin"].map("{:.1f}%".format)
print(summary.to_string())


## Univariate Analysis

In [ ]:
band_stats = df.groupby("Discount Band", observed=False).agg(
    Orders      = ("Order ID", "count"),
    Total_Sales = ("Sales", "sum"),
    Total_Profit= ("Profit", "sum"),
    Units       = ("Quantity", "sum"),
    Avg_Margin  = ("Margin Pct", "mean"),
).reindex(DISCOUNT_LABELS)

band_stats["Margin %"] = (band_stats["Total_Profit"] / band_stats["Total_Sales"] * 100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = sns.color_palette("RdYlGn", len(DISCOUNT_LABELS))

# Revenue
axes[0].bar(band_stats.index, band_stats["Total_Sales"]/1e6, color=palette)
axes[0].set_title("Total Revenue by Discount Band")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:.1f}M"))
axes[0].set_xlabel("Discount Band"); axes[0].tick_params(axis="x", rotation=15)

# Profit
axes[1].bar(band_stats.index, band_stats["Total_Profit"]/1e3, color=palette)
axes[1].set_title("Total Profit by Discount Band")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:.0f}K"))
axes[1].set_xlabel("Discount Band"); axes[1].tick_params(axis="x", rotation=15)

# Margin %
colors = ["tomato" if m < 0 else "#4878d0" for m in band_stats["Margin %"]]
axes[2].bar(band_stats.index, band_stats["Margin %"], color=colors)
axes[2].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[2].set_title("Profit Margin % by Discount Band")
axes[2].set_ylabel("Margin %"); axes[2].set_xlabel("Discount Band")
axes[2].tick_params(axis="x", rotation=15)
for bar, v in zip(axes[2].patches, band_stats["Margin %"]):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+(0.5 if v>=0 else -1.5),
                 f"{v:.1f}%", ha="center", fontsize=8)

plt.suptitle("Revenue, Profit & Margin Across Discount Bands", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "band_revenue_margin.png", bbox_inches="tight")
plt.show()
print(band_stats[["Orders","Total_Sales","Total_Profit","Units","Margin %"]].to_string())


## Bivariate / Multivariate Analysis
Each point is one order line. We fit a linear trend to quantify how much margin is
lost per additional percentage point of discount.


In [ ]:
sample = df.sample(min(3000, len(df)), random_state=SEED)

slope, intercept, r, p, se = stats.linregress(sample["Discount"], sample["Margin Pct"])

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(sample["Discount"], sample["Margin Pct"],
           alpha=0.25, s=15, c="#4878d0", label="Order lines")

x_line = np.linspace(0, sample["Discount"].max(), 200)
ax.plot(x_line, intercept + slope*x_line, color="tomato", linewidth=2.5,
        label=f"Trend: {slope:+.1f}pp margin / 1pp discount  (R²={r**2:.3f})")

ax.axhline(0, color="black", linewidth=0.8, linestyle="--", label="Break-even")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xlabel("Discount"); ax.set_ylabel("Profit Margin %")
ax.set_title("Discount Depth vs Profit Margin per Order Line")
ax.legend(); ax.set_ylim(-300, 200)
plt.tight_layout()
plt.savefig(OUT_DIR / "discount_vs_margin_scatter.png", bbox_inches="tight")
plt.show()

print(f"Regression: margin = {intercept:.1f}% + ({slope:.2f}) × discount")
print(f"R² = {r**2:.4f}  |  p-value = {p:.4e}")
print(f"Interpretation: each 10pp increase in discount → {slope*0.10:.1f}pp margin change")


## 8. Does Discounting Drive Volume?

Bin orders by discount and check if average quantity per order increases.


In [ ]:
volume_stats = df.groupby("Discount Band", observed=False).agg(
    Avg_Qty   = ("Quantity", "mean"),
    Median_Qty= ("Quantity", "median"),
    Orders    = ("Order ID", "count"),
).reindex(DISCOUNT_LABELS)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(DISCOUNT_LABELS))
bars = ax.bar(x, volume_stats["Avg_Qty"],
              color=sns.color_palette("Blues_d", len(DISCOUNT_LABELS)))
ax.plot(x, volume_stats["Median_Qty"], "o--", color="tomato",
        label="Median Qty", linewidth=2, markersize=8)
ax.set_xticks(x); ax.set_xticklabels(DISCOUNT_LABELS, rotation=10)
ax.set_title("Average & Median Order Quantity by Discount Band")
ax.set_ylabel("Quantity per Order")
for bar, v in zip(bars, volume_stats["Avg_Qty"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            f"{v:.2f}", ha="center", fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "discount_volume_effect.png", bbox_inches="tight")
plt.show()
print(volume_stats.to_string())


## Feature-Specific Insights
How do the three product categories respond differently to discounts?


In [ ]:
cat_band = df.groupby(["Category", "Discount Band"], observed=False).agg(
    Orders    = ("Order ID", "count"),
    Revenue   = ("Sales", "sum"),
    Profit    = ("Profit", "sum"),
    Avg_Margin= ("Margin Pct", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
categories = df["Category"].unique()
palette = sns.color_palette("tab10", len(categories))

for i, (cat, ax) in enumerate(zip(categories, axes)):
    sub = cat_band[cat_band["Category"] == cat].set_index("Discount Band").reindex(DISCOUNT_LABELS)
    colors = ["tomato" if m < 0 else "#4878d0" for m in sub["Avg_Margin"].fillna(0)]
    ax.bar(DISCOUNT_LABELS, sub["Avg_Margin"].fillna(0), color=colors)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(f"{cat}")
    ax.set_xlabel("Discount Band"); ax.set_ylabel("Avg Margin %" if i==0 else "")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Profit Margin % by Discount Band — Per Category", fontsize=14,
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "category_discount_margin.png", bbox_inches="tight")
plt.show()


## Statistical ChecksThese checks compare discounted and non-discounted orders with non-parametric tests, which is safer for skewed retail order distributions than assuming normality.

In [ ]:
from scipy import statsdisc_profit = df.loc[df['Is Discounted'], 'Profit'].dropna()no_disc_profit = df.loc[~df['Is Discounted'], 'Profit'].dropna()disc_sales = df.loc[df['Is Discounted'], 'Sales'].dropna()no_disc_sales = df.loc[~df['Is Discounted'], 'Sales'].dropna()u_profit, p_profit = stats.mannwhitneyu(disc_profit, no_disc_profit, alternative='two-sided')u_sales, p_sales = stats.mannwhitneyu(disc_sales, no_disc_sales, alternative='two-sided')print(f'Mann-Whitney test on Profit (discounted vs non-discounted): U={u_profit:.0f}, p={p_profit:.2e}')print(f'Mann-Whitney test on Sales (discounted vs non-discounted): U={u_sales:.0f}, p={p_sales:.2e}')

## Visual Analysis
Reveals which sub-categories are most sensitive to discounting.


In [ ]:
pivot = df.pivot_table(
    values="Margin Pct",
    index="Sub-Category",
    columns="Discount Band",
    aggfunc="mean",
    observed=False
).reindex(columns=DISCOUNT_LABELS)

fig, ax = plt.subplots(figsize=(14, 9))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn", center=0,
            linewidths=0.5, ax=ax,
            cbar_kws={"label": "Avg Profit Margin %"})
ax.set_title("Average Profit Margin % by Sub-Category × Discount Band")
ax.set_xlabel("Discount Band"); ax.set_ylabel("Sub-Category")
plt.tight_layout()
plt.savefig(OUT_DIR / "subcategory_discount_heatmap.png", bbox_inches="tight")
plt.show()


## 11. Revenue Leakage Estimation

How much additional revenue would have been generated if discounts were capped at 20%?


In [ ]:
df["Imputed_Sales_No_Disc"] = df["Sales"] / (1 - df["Discount"].clip(upper=0.999))
df["Imputed_Sales_Cap20"]   = df["Sales"] / (1 - df["Discount"].clip(upper=0.20).clip(lower=0))

actual_rev  = df["Sales"].sum()
nodisc_rev  = df["Imputed_Sales_No_Disc"].sum()
cap20_rev   = df["Imputed_Sales_Cap20"].sum()

leakage_nodisc = nodisc_rev - actual_rev
leakage_cap20  = cap20_rev  - actual_rev

print("=" * 55)
print(f"Actual recorded revenue        : ${actual_rev:>12,.0f}")
print(f"Revenue if no discounts given  : ${nodisc_rev:>12,.0f}  (+${leakage_nodisc:,.0f})")
print(f"Revenue if discounts capped@20%: ${cap20_rev:>12,.0f}  (+${leakage_cap20:,.0f})")
print("=" * 55)
print()
print(f"Estimated revenue leakage from deep discounts (>20%): ${leakage_cap20 - 0:,.0f}")

# by category
cat_leak = df[df["Discount"] > 0.20].groupby("Category").agg(
    Lost_Revenue = ("Sales", lambda x: (
        (x / (1 - df.loc[x.index,"Discount"].clip(upper=0.999))) - x
        ).sum())
).sort_values("Lost_Revenue", ascending=False)
print()
print("Revenue leakage by category (orders where discount > 20%):")
print(cat_leak.to_string())


## 12. Seasonality of Discounting

Does the business discount more during certain months / quarters?


In [ ]:
monthly = df.groupby(["Year","Month"]).agg(
    Avg_Discount = ("Discount", "mean"),
    Disc_Share   = ("Is Discounted", "mean"),  # fraction of orders discounted
    Revenue      = ("Sales", "sum"),
    Avg_Margin   = ("Margin Pct", "mean"),
).reset_index()

# pivot for heatmap
pivot_disc = monthly.pivot(index="Month", columns="Year", values="Avg_Discount")
pivot_marg = monthly.pivot(index="Month", columns="Year", values="Avg_Margin")

month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(pivot_disc, annot=True, fmt=".1%", cmap="YlOrRd",
            xticklabels=True, yticklabels=month_labels, ax=axes[0],
            cbar_kws={"label": "Avg Discount"})
axes[0].set_title("Average Discount % by Month × Year")
axes[0].set_xlabel("Year"); axes[0].set_ylabel("Month")

sns.heatmap(pivot_marg, annot=True, fmt=".1f", cmap="RdYlGn", center=0,
            xticklabels=True, yticklabels=month_labels, ax=axes[1],
            cbar_kws={"label": "Margin %"})
axes[1].set_title("Avg Profit Margin % by Month × Year")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("Month")

plt.suptitle("Seasonal Discounting Patterns", fontsize=14,
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "seasonal_discount_heatmap.png", bbox_inches="tight")
plt.show()


## 13. Identifying the Optimal Discount Zone

Plot average margin vs average revenue per order at each discount level
to find the zone where revenue is high and margin remains positive.


In [ ]:
fine = df.copy()
fine["Disc_Pct"] = (fine["Discount"] * 100).round(0).astype(int)

zone = fine.groupby("Disc_Pct").agg(
    Orders    = ("Order ID", "count"),
    Avg_Rev   = ("Sales", "mean"),
    Avg_Margin= ("Margin Pct", "mean"),
    Avg_Profit= ("Profit", "mean"),
).reset_index()
zone = zone[zone["Orders"] >= 10]  # only statistically meaningful bands

fig, ax = plt.subplots(figsize=(11, 6))
sc = ax.scatter(zone["Avg_Rev"], zone["Avg_Margin"],
                c=zone["Disc_Pct"], cmap="RdYlGn_r",
                s=zone["Orders"]/zone["Orders"].max()*400 + 30,
                alpha=0.8, edgecolors="grey", linewidth=0.4)
plt.colorbar(sc, ax=ax, label="Discount %")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--", label="Break-even margin")
ax.axvline(zone["Avg_Rev"].median(), color="#4878d0", linewidth=0.8,
           linestyle=":", label="Median revenue")

# annotate key discount levels
for _, row in zone[zone["Disc_Pct"].isin([0,10,20,30,40,50])].iterrows():
    ax.annotate(f"{int(row['Disc_Pct'])}%",
                (row["Avg_Rev"], row["Avg_Margin"]),
                textcoords="offset points", xytext=(5,5), fontsize=9)

ax.set_xlabel("Average Revenue per Order ($)")
ax.set_ylabel("Average Profit Margin %")
ax.set_title("Discount Zone Analysis: Revenue vs Margin Trade-off
(bubble size ∝ order count)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "optimal_discount_zone.png", bbox_inches="tight")
plt.show()

# optimal = positive margin, top-quartile revenue
threshold_rev    = zone["Avg_Rev"].quantile(0.50)
optimal = zone[(zone["Avg_Margin"] > 0) & (zone["Avg_Rev"] >= threshold_rev)]
print("Discount levels with POSITIVE margin and ABOVE-MEDIAN revenue:")
print(optimal[["Disc_Pct","Orders","Avg_Rev","Avg_Margin","Avg_Profit"]].to_string(index=False))


## 14. Customer Segment Sensitivity to Discounts

Which segments (Consumer, Corporate, Home Office) show the sharpest margin
drop under discounts?


In [ ]:
seg_band = df.groupby(["Segment", "Discount Band"], observed=False).agg(
    Orders    = ("Order ID", "count"),
    Revenue   = ("Sales", "sum"),
    Avg_Margin= ("Margin Pct", "mean"),
).reset_index()

pivot_seg = seg_band.pivot(index="Segment", columns="Discount Band", values="Avg_Margin")
pivot_seg = pivot_seg.reindex(columns=DISCOUNT_LABELS)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot_seg, annot=True, fmt=".1f", cmap="RdYlGn", center=0,
            linewidths=0.6, ax=ax, cbar_kws={"label": "Avg Margin %"})
ax.set_title("Profit Margin % by Segment × Discount Band")
ax.set_xlabel("Discount Band"); ax.set_ylabel("Segment")
plt.tight_layout()
plt.savefig(OUT_DIR / "segment_discount_heatmap.png", bbox_inches="tight")
plt.show()


## 15. Discount ROI — Does the Volume Lift Justify the Margin Cost?

For each discount band, compare:
- **Revenue lift** from higher sales volume relative to the no-discount baseline
- **Margin cost** of the price reduction

If Volume Lift % < Margin Cost %, discounting is value-destroying.


In [ ]:
base = df[df["Discount"] == 0]
base_avg_rev  = base["Sales"].mean()
base_avg_marg = base["Margin Pct"].mean()

roi_table = df.groupby("Discount Band", observed=False).agg(
    Avg_Rev   = ("Sales", "mean"),
    Avg_Margin= ("Margin Pct", "mean"),
    Orders    = ("Order ID", "count"),
).reindex(DISCOUNT_LABELS)

roi_table["Rev Lift vs No-Discount (%)"] = (
    (roi_table["Avg_Rev"] - base_avg_rev) / abs(base_avg_rev) * 100
).round(1)
roi_table["Margin Drop vs No-Discount (pp)"] = (
    roi_table["Avg_Margin"] - base_avg_marg
).round(1)
roi_table["Value Creating?"] = roi_table.apply(
    lambda r: "✓ Yes" if r["Rev Lift vs No-Discount (%)"] > 0 and r["Avg_Margin"] > 0
              else "✗ Destroys Value", axis=1
)

print("Discount ROI Summary")
print("=" * 80)
print(roi_table[["Orders","Avg_Rev","Avg_Margin",
                 "Rev Lift vs No-Discount (%)","Margin Drop vs No-Discount (pp)",
                 "Value Creating?"]].to_string())


## 16. Executive Summary Statistics


## Common Mistakes- Judging discount success by sales alone without checking profit.- Assuming every category benefits equally from discount depth.- Ignoring skewed order distributions when comparing discounted and non-discounted orders.- Treating all revenue lift as incremental rather than timing-shifted demand.- Forgetting that some products cannot absorb deep discounts profitably.

## Recommendations / Next Steps1. Define category-specific discount guardrails based on profit tolerance.2. Separate tactical promotions from recurring deep-discount habits.3. Re-run this analysis after adding campaign, inventory, and competitor context.4. Test segment-specific discount rules instead of one uniform policy.

## Final Summary / Key TakeawaysDiscounts can increase demand, but the commercial test is whether the added sales justify the margin cost. This notebook keeps the analysis grounded in profit, category mix, and segment behaviour so promotion decisions remain economically defensible.

In [ ]:
disc_orders   = (df["Is Discounted"].sum() / len(df) * 100)
avg_disc_when = df[df["Is Discounted"]]["Discount"].mean()
disc_rev_share= (df[df["Is Discounted"]]["Sales"].sum() / df["Sales"].sum() * 100)
disc_pft_share= (df[df["Is Discounted"]]["Profit"].sum() / df["Profit"].sum() * 100)
avg_marg_disc = df[df["Is Discounted"]]["Margin Pct"].mean()
avg_marg_nond = df[~df["Is Discounted"]]["Margin Pct"].mean()

print("=" * 55)
print("DISCOUNT EFFECTIVENESS — EXECUTIVE SUMMARY")
print("=" * 55)
print(f"Orders that carry a discount      : {disc_orders:.1f}%")
print(f"Average discount depth (when used): {avg_disc_when:.1%}")
print(f"Discounted orders' revenue share  : {disc_rev_share:.1f}%")
print(f"Discounted orders' profit share   : {disc_pft_share:.1f}%")
print(f"Avg margin — discounted orders    : {avg_marg_disc:.1f}%")
print(f"Avg margin — non-discounted orders: {avg_marg_nond:.1f}%")
print(f"Margin gap (discounted vs not)    : {avg_marg_disc - avg_marg_nond:.1f}pp")
print()
print("Worst offender sub-categories (most negative margin under discount):")
worst = (df[df["Discount"] > 0.20]
         .groupby("Sub-Category")["Margin Pct"].mean()
         .sort_values()
         .head(5))
for sc, m in worst.items():
    print(f"  {sc:<25} {m:+.1f}%")


## Key Findings
### Key Findings

1. **Discounting does NOT drive proportional volume lift** — average order quantity
   stays roughly flat across discount bands, meaning the company is giving away margin
   without a compensating unit increase.

2. **Margin collapses beyond ~20%** — the linear regression confirms a steep negative
   relationship between discount depth and profit margin. Orders above 20% discount
   frequently show negative margins.

3. **Furniture & Technology** are most harmed by deep discounts. Office Supplies can
   tolerate moderate discounts (1-20%) while still retaining positive margins.

4. **Seasonal discounting patterns** coincide with Q4 promotions but the margin
   impact outweighs the revenue gain in most categories during those months.

5. **High-discount orders (>30%) represent a disproportionate share of total
   losses** — removing or capping them would meaningfully improve overall profitability.

### Recommendations

| Priority | Action | Expected Impact |
|---|---|---|
| High | Cap maximum discount at 20% company-wide | Eliminate negative-margin orders |
| High | Restrict deep discounts on Furniture (Tables, Bookcases) | These sub-categories lose money even at moderate discounts |
| Medium | Replace blanket discounts with volume-based incentives | Reward large orders without blanket price cuts |
| Medium | Pilot "no-discount" months in low-competition regions | Validate price elasticity assumption |
| Low | Build a discount approval workflow for >15% requests | Add friction to high-cost discounting decisions |

### Limitations

- Dataset spans 2014-2017; competitive pricing conditions may have changed.
- Returns are not modelled here; returned discounted goods add further cost.
- The Superstore dataset does not capture customer lifetime value — a discount that
  wins a long-term contract may still be rational even at short-term margin loss.


## 18. Mini Challenge

1. Fit a non-linear (polynomial or LOWESS) curve to the discount-vs-margin scatter
   to see if there is a non-linear threshold.
2. Build a segment-aware discount policy: calculate the maximum discount each customer
   segment can receive while keeping avg margin ≥ 5%.
3. Simulate the P&L impact of a strict 20% discount cap across all historical orders.


---
*Notebook: Discount Effectiveness Analysis*
*Dataset: Sample Superstore (Orders sheet)*
*Techniques: Discount banding, regression, heatmaps, ROI decomposition*
